In [ ]:
# If needed: pip install chromadb sentence-transformers
# !pip install chromadb sentence-transformers

import chromadb
from chromadb.utils import embedding_functions

DB_PATH = "C:\\Users\\shubh\\OneDrive\\Documents\\Tutorials\\FilmGPT\\FilmGPT\\data\\chroma_db"

client = chromadb.PersistentClient(path=DB_PATH)

print("Available collections:")
for coll in client.list_collections():
    print(f"- {coll.name}")

Available collections:
- tmdb_synopsis
- letterboxd_personal


In [3]:
# Recreate the embedding function used to build the collections
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"  # as in your ingestion scripts
)

# Load the two main collections
tmdb_collection = client.get_collection(
    name="tmdb_synopsis",
    embedding_function=embedding_fn,
)

letterboxd_collection = client.get_collection(
    name="letterboxd_personal",
    embedding_function=embedding_fn,
)

print("tmdb_synopsis count:", tmdb_collection.count())
print("letterboxd_personal count:", letterboxd_collection.count())

c:\Users\shubh\anaconda3\envs\filmgpt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3237.12it/s]


tmdb_synopsis count: 898
letterboxd_personal count: 905


In [4]:
# Inspect a few TMDb synopsis docs
tmdb_sample = tmdb_collection.get(limit=3)
print("TMDb sample ids:", tmdb_sample.get("ids"))
print("TMDb sample metadatas[0]:", tmdb_sample.get("metadatas")[0])
print("TMDb sample documents[0]:")
print(tmdb_sample.get("documents")[0])

# Inspect a few personal Letterboxd docs
lb_sample = letterboxd_collection.get(limit=3)
print("\nLetterboxd sample ids:", lb_sample.get("ids"))
print("Letterboxd sample metadatas[0]:", lb_sample.get("metadatas")[0])
print("Letterboxd sample documents[0]:")
print(lb_sample.get("documents")[0])

TMDb sample ids: ['tmdb_19913', 'tmdb_389', 'tmdb_76203']
TMDb sample metadatas[0]: {'tmdb_vote_average': 7.3, 'film_key': '(500) days of summer:2009', 'runtime_minutes': 95, 'directors': 'Marc Webb', 'letterboxd_rating': 4.5, 'tmdb_id': 19913, 'year': 2009, 'genres': 'Comedy|Drama|Romance', 'title': '(500) Days of Summer'}
TMDb sample documents[0]:
(500) Days of Summer (2009)
Overview: Tom, greeting-card writer and hopeless romantic, is caught completely off-guard when his girlfriend, Summer, suddenly dumps him. He reflects on their 500 days together to try to figure out where their love affair went sour, and in doing so, Tom rediscovers his true passions in life.
Genres: Comedy, Drama, Romance.
Directed by: Marc Webb.
Starring: Joseph Gordon-Levitt, Zooey Deschanel, Geoffrey Arend, Chloë Grace Moretz, Matthew Gray Gubler.
Keywords: jealousy, gallery, fight, date, architect, interview, romcom, sister, love, friends, fate, los angeles, california.
Similar films: Opera Prima, Jamon Jamo

In [5]:
# Try a query that should clearly hit some films in your collection
tmdb_query_result = tmdb_collection.query(
    query_texts=[
        "psychological thriller about a serial killer investigation",
    ],
    n_results=5,
)

print("TMDb query result ids:", tmdb_query_result.get("ids"))
print("TMDb query result metadatas:")
for md in tmdb_query_result.get("metadatas"):
    print(md)
print("\nTMDb query result documents[0]:")
print(tmdb_query_result.get("documents")[0])

TMDb query result ids: [['tmdb_11423', 'tmdb_146233', 'tmdb_215', 'tmdb_69775', 'tmdb_93']]
TMDb query result metadatas:
[{'directors': 'Bong Joon Ho', 'tmdb_id': 11423, 'tmdb_vote_average': 8.062, 'title': 'Memories of Murder', 'letterboxd_rating': 4.5, 'film_key': 'memories of murder:2003', 'runtime_minutes': 131, 'year': 2003, 'genres': 'Crime|Drama|Thriller'}, {'tmdb_id': 146233, 'directors': 'Denis Villeneuve', 'genres': 'Drama|Thriller|Crime', 'runtime_minutes': 153, 'year': 2013, 'film_key': 'prisoners:2013', 'title': 'Prisoners', 'tmdb_vote_average': 8.1}, {'tmdb_id': 215, 'directors': 'Darren Lynn Bousman', 'genres': 'Horror', 'film_key': 'saw ii:2005', 'runtime_minutes': 93, 'tmdb_vote_average': 6.608, 'title': 'Saw II', 'year': 2005}, {'year': 2011, 'tmdb_id': 69775, 'tmdb_vote_average': 5.758, 'title': 'Murder 2', 'film_key': 'murder 2:2011', 'directors': 'Mohit Suri', 'genres': 'Action|Crime|Drama|Thriller', 'runtime_minutes': 127}, {'film_key': 'anatomy of a murder:1959',

In [ ]:
# Now confirm the personal‑taste collection responds with your ratings/reviews
lb_query_result = letterboxd_collection.query(
    query_texts=[
        "films I loved with dark psychological themes",
    ],
    n_results=5,
)

print("Letterboxd query result ids:", lb_query_result.get("ids"))
print("Letterboxd query result metadatas:")
for md in lb_query_result.get("metadatas"):
    print(md)
print("\nLetterboxd query result documents[0]:")
print(lb_query_result.get("documents")[0])

Letterboxd query result ids: [['letterboxd_the mummy:1999', 'letterboxd_little manhattan:2005', 'letterboxd_high and low:1963', 'letterboxd_the ring:2002', 'letterboxd_snatch:2000']]
Letterboxd query result metadatas:
[{'film_key': 'the mummy:1999', 'watched': True, 'letterboxd_uri': 'https://boxd.it/2a7a', 'year': 1999, 'title': 'The Mummy'}, {'letterboxd_uri': 'https://boxd.it/1JXS', 'year': 2005, 'title': 'Little Manhattan', 'film_key': 'little manhattan:2005', 'watched': True}, {'film_key': 'high and low:1963', 'title': 'High and Low', 'year': 1963, 'letterboxd_rating': 5.0, 'letterboxd_uri': 'https://boxd.it/1RSc', 'watched': True}, {'letterboxd_uri': 'https://boxd.it/2a70', 'watched': True, 'title': 'The Ring', 'film_key': 'the ring:2002', 'year': 2002}, {'year': 2000, 'letterboxd_uri': 'https://boxd.it/2b7U', 'film_key': 'snatch:2000', 'title': 'Snatch', 'watched': True}]

Letterboxd query result documents[0]:
['The Mummy (1999)\nMy rating: not rated.\nWatched: yes. No diary dat

In [8]:
# Try a query that should clearly hit some films in your collection
tmdb_query_result = tmdb_collection.query(
    query_texts=[
        "give me a short summry of the taxi driver",
    ],
    n_results=5,
)

print("TMDb query result ids:", tmdb_query_result.get("ids"))
print("TMDb query result metadatas:")
for md in tmdb_query_result.get("metadatas"):
    print(md)
print("\nTMDb query result documents[0]:")
print(tmdb_query_result.get("documents")[0])

TMDb query result ids: [['tmdb_103', 'tmdb_339403', 'tmdb_1538', 'tmdb_64690', 'tmdb_1455945']]
TMDb query result metadatas:
[{'title': 'Taxi Driver', 'letterboxd_rating': 5.0, 'runtime_minutes': 114, 'genres': 'Crime|Drama', 'film_key': 'taxi driver:1976', 'year': 1976, 'tmdb_vote_average': 8.122, 'tmdb_id': 103, 'directors': 'Martin Scorsese'}, {'title': 'Baby Driver', 'tmdb_id': 339403, 'directors': 'Edgar Wright', 'film_key': 'baby driver:2017', 'tmdb_vote_average': 7.444, 'year': 2017, 'genres': 'Action|Crime', 'runtime_minutes': 113}, {'year': 2004, 'genres': 'Drama|Crime|Thriller', 'tmdb_vote_average': 7.247, 'runtime_minutes': 120, 'title': 'Collateral', 'directors': 'Michael Mann', 'tmdb_id': 1538, 'film_key': 'collateral:2004'}, {'title': 'Drive', 'tmdb_id': 64690, 'runtime_minutes': 100, 'directors': 'Nicolas Winding Refn', 'year': 2011, 'film_key': 'drive:2011', 'tmdb_vote_average': 7.583, 'genres': 'Drama|Thriller|Crime'}, {'year': 2014, 'title': 'Highway', 'tmdb_id': 1455